In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from config import DATA_DIR

In [ ]:
df_parkings = pd.read_parquet(DATA_DIR / 'parkings.parquet')

In [ ]:
df_parkings_availabilities = pd.read_parquet(DATA_DIR / 'parking_availabilities.parquet')

In [ ]:
df_sks_users = pd.read_parquet(DATA_DIR / 'sks_users.parquet')

In [ ]:
df_events = pd.read_parquet(DATA_DIR / 'pwr_events.parquet')

In [ ]:
pd.set_option('display.max_columns',30)
pd.set_option('display.max_rows',288)

# Events EDA

In [ ]:
df_events.head(50)

In [ ]:
df_events.shape

In [ ]:
missing_location = df_events['location'][df_events['location'] == ""].count()
print(f"How many events don't have a location: {missing_location}")

In [ ]:
percent_missing = (missing_location/df_events.shape[0]) * 100
print(f"This means {percent_missing}% of events don't have a specified location")

In [ ]:
df_events['location'].value_counts()

Using NLP to extract information might be pointless, especially because most of location names are unique.  

# Parkings EDA

In [ ]:
df_parkings.head(10)

In [ ]:
df_parkings_availabilities = df_parkings_availabilities.sort_values(by=['measured_at'])
df_parkings_availabilities = df_parkings_availabilities.reset_index(drop=True)
df_parkings_availabilities.head(100)

In [ ]:
df_parkings_availabilities.shape

2 million rows - propably should downsample that to a day instead of minute for some plots

### Adding features

In [ ]:
df_parkings_availabilities['day'] = df_parkings_availabilities['measured_at'].dt.day_name()

df_parkings_availabilities[['measured_at', 'day']].tail()

In [ ]:
def is_open_parking(row):
    if row['parking_id'] == 2:
        return True
    
    time = row['measured_at'].time()
    if row['parking_id'] == 4:
        # check if between 6:00 and 22:00
        if dt.time(6, 0) <= time < dt.time(22, 0):
            return True
        else:
            return False
    else:
        # check if between 6:00 and 22:30
        if dt.time(6, 0) <= time < dt.time(22, 30):
            return True
        else:
            return False
    

In [ ]:
df_parkings_availabilities['is_open'] = df_parkings_availabilities.apply(is_open_parking, axis=1)

In [ ]:
event_intervals = pd.IntervalIndex.from_arrays(
    df_events['start'], 
    df_events['end'], 
)
df_parkings_availabilities['has_event'] = df_parkings_availabilities['measured_at'].apply(lambda x: event_intervals.contains(x).any())

In [ ]:
df_parkings_availabilities['has_event'].value_counts()

### Check data anomlaites, duplicates, etc. 

In [ ]:
time_cols = ['created_at', 'measured_at', 'updated_at']

time_spread = df_parkings_availabilities[time_cols].max(axis=1) - df_parkings_availabilities[time_cols].min(axis=1)

anomalies = df_parkings_availabilities[time_spread > pd.Timedelta(seconds=60)]
print(f"Found {len(anomalies)} anomalies where time gap > 60s.")
anomalies.head(20)

There are 779 rows witch seem to be anomalies in this dataframe. Delta of over 60 seconds is problematic when values are logged every 60 seconds. Proposed soution is dropping those problematic rows and imputing missing values with knn encoders or similar. There seems to be enough "good" data to fill the gpas without issues. Another possible solution is keeping those rows and adding anomally flag for them. Intresting thing is anomalies in time delta seem to occur across all parkings at the same time.

In [ ]:
value_counts = df_parkings_availabilities["updated_at"].value_counts()
duplicates = value_counts[value_counts > 1]
print(duplicates)

In [ ]:
value_counts = df_parkings_availabilities["created_at"].value_counts()
duplicates = value_counts[value_counts > 1]
print(duplicates)

In [ ]:
are_identical = df_parkings_availabilities["created_at"].equals(df_parkings_availabilities["updated_at"])

print(f"Columns are identical: {are_identical}")

Even tho columns "updated_at" and "created_at" poses no duplicate values individually, they often store same values, their usablity seems low as they very closely copy "measured_at" so they should be dropped. Although the time delta between "measured_at" and "created_at" or "updated_at" informs us of delay in infiormation logging but it's so small it's neglectable, can be used for anomally detection tho. 

### Data resampling 

In [ ]:
df_parking_architektura = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 7]
df_parking_architektura = df_parking_architektura.reset_index(drop=True)

df_parking_wronskiego = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 4]
df_parking_wronskiego = df_parking_wronskiego.reset_index(drop=True)

df_parking_polinka = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 2]
df_parking_polinka = df_parking_polinka.reset_index(drop=True)

df_parking_D20 = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 5]
df_parking_D20 = df_parking_D20.reset_index(drop=True)

df_parking_geocentrum = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 6]
df_parking_geocentrum = df_parking_geocentrum.reset_index(drop=True)

df_parking_architektura.head(288)

In [ ]:

def check_missing_logs(df_to_check, time_column_name, time_between_logs = 60, allowed_treshold = 10):
    df_to_check = df_to_check.sort_values(time_column_name)
    df_to_check['time_since_prev'] = df_to_check[time_column_name].diff()
    
    threshold =  pd.Timedelta(seconds=allowed_treshold+time_between_logs)
    
    gaps = df_to_check[df_to_check['time_since_prev'] > threshold]
    
    print(f"Found {len(gaps)} gaps in data logging.")
    print(gaps[[time_column_name, 'time_since_prev']].sort_values(by=['time_since_prev'],  ascending=False).head(5))

In [ ]:
check_missing_logs(df_parking_architektura, 'measured_at')

In [ ]:
check_missing_logs(df_parking_wronskiego, 'measured_at')

In [ ]:
check_missing_logs(df_parking_polinka, 'measured_at')

In [ ]:
check_missing_logs(df_parking_D20, 'measured_at')

In [ ]:
check_missing_logs(df_parking_geocentrum, 'measured_at')

Gaps are consistent across all parkings witch indicates loggin issiue rather than individual sensor failure. Some gaps are quite large few days long but they probably could be imputed. 

In [ ]:
df_parking_architektura.iloc[25900 :26000]  

In [ ]:
agg_rules = {
    'spaces_left': 'mean',
    'trend': 'mean',
    'parking_id': 'first',
    'day': 'first', 
    'is_open': 'max',
    'has_event': 'max'
}
df_parking_architektura_5min = df_parking_architektura.resample('5min', on='measured_at').agg(agg_rules)
df_parking_wronskiego_5min = df_parking_wronskiego.resample('5min', on='measured_at').agg(agg_rules)
df_parking_polinka_5min = df_parking_polinka.resample('5min', on='measured_at').agg(agg_rules)
df_parking_D20_5min = df_parking_D20.resample('5min', on='measured_at').agg(agg_rules)
df_parking_geocentrum_5min = df_parking_geocentrum.resample('5min', on='measured_at').agg(agg_rules)


In [ ]:
df_parking_architektura_5min.head(20)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(50, 50), sharex=True)

sns.lineplot(data=df_parking_architektura_5min, x="measured_at", y="spaces_left", hue="is_open", ax=axes[0])
axes[0].set_title("Parking architektura is_open")

sns.lineplot(data=df_parking_wronskiego_5min, x="measured_at", y="spaces_left", hue="is_open", ax=axes[1])
axes[1].set_title("Parking wronskiego is_open")

sns.lineplot(data=df_parking_polinka_5min, x="measured_at", y="spaces_left", hue="is_open", ax=axes[2])
axes[2].set_title("Parking polinka is_open")

sns.lineplot(data=df_parking_D20_5min, x="measured_at", y="spaces_left", hue="is_open", ax=axes[3])
axes[3].set_title("Parking D20 is_open")

sns.lineplot(data=df_parking_geocentrum_5min, x="measured_at", y="spaces_left", hue="is_open", ax=axes[4])
axes[4].set_title("Parking geocentruym is_open")

plt.show()

In [ ]:
parking_lots = [
    ("Parking Architektura", df_parking_architektura_5min),
    ("Parking Wronskiego", df_parking_wronskiego_5min),
    ("Parking Polinka", df_parking_polinka_5min),
    ("Parking D20", df_parking_D20_5min),
    ("Parking Geocentrum", df_parking_geocentrum_5min)
]

fig = make_subplots(
    rows=len(parking_lots), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=[item[0] for item in parking_lots], # item[0] is the title
    vertical_spacing=0.08
)

for i, (title, df) in enumerate(parking_lots):

    dff = df.copy()
    dff["is_open"] = dff["is_open"].astype(str)
    
    temp_fig = px.line(
        dff, 
        x=dff.index, 
        y="spaces_left", 
        color="is_open",
        color_discrete_map={"True": "red", "False": "blue"}
    )
    

    for trace in temp_fig.data:
        trace.legendgroup = trace.name 
        
        if i > 0:
            trace.showlegend = False
            
        fig.add_trace(trace, row=i+1, col=1)


fig.update_layout(
    height=1500,  
    width=1500,
    title_text="Parking Availability Analysis",
    hovermode="x unified"
)

fig.update_xaxes(title_text="Measured At", row=5, col=1)

fig.show()

From the charts we can see few anomalies: parking lots "Polinka" and "Wronskiego" have values below 0. It suggests that during this time more cars managed to park than would seem from designated parking spaces. Also, it could have been motorbikes or similar. Two or more motorbikes can easily fit inside one normal parking space. Additionally, we can observe that "wronskiego" parking lot was expanded in May.  is_open seems to nicely correlate with heavier traffic. 

In [ ]:
fig = make_subplots(
    rows=len(parking_lots), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=[item[0] for item in parking_lots], # item[0] is the title
    vertical_spacing=0.08
)

for i, (title, df) in enumerate(parking_lots):

    dff = df.copy()
    dff["has_event"] = dff["has_event"].astype(str)
    
    temp_fig = px.line(
        dff, 
        x=dff.index, 
        y="spaces_left", 
        color="has_event",
        color_discrete_map={"True": "red", "False": "blue"}
    )
    

    for trace in temp_fig.data:
        trace.legendgroup = trace.name 
        
        if i > 0:
            trace.showlegend = False
            
        fig.add_trace(trace, row=i+1, col=1)


fig.update_layout(
    height=1500,  
    width=1500,
    title_text="Parking Availability Analysis",
    hovermode="x unified"
)

fig.update_xaxes(title_text="Measured At", row=5, col=1)

fig.show()

has_event in current form does not seem to be usefull it rarely correlates with aviable parking space in meaningfull way. 

In [ ]:
def add_lag_parkings(df):
    df['lag_1_hour'] = df['spaces_left'].shift(12) # 12 * 5 = 60 min 
    df['lag_3_hours'] = df['spaces_left'].shift(36)
    df['lag_1_day'] = df['spaces_left'].shift(288) # 24 * 60 / 5 = 288
    df['lag_7_days'] = df['spaces_left'].shift(288*7)
    df['lag_2_weeks'] = df['spaces_left'].shift(288*7*2)
    return df

In [ ]:
df_parking_architektura_5min = add_lag_parkings(df_parking_architektura_5min)
df_parking_wronskiego_5min = add_lag_parkings(df_parking_wronskiego_5min)
df_parking_polinka_5min = add_lag_parkings(df_parking_polinka_5min)
df_parking_D20_5min = add_lag_parkings(df_parking_D20_5min)
df_parking_geocentrum_5min = add_lag_parkings(df_parking_geocentrum_5min)

In [ ]:
df_parking_architektura_5min.head(5)

In [ ]:
def add_other_features(df):
    df['roll_mean_3h'] = df['spaces_left'].rolling(window=36, closed='left').mean()
    df['roll_mean_24h'] = df['spaces_left'].rolling(window=288, closed='left').mean()

    df['delta_1h'] = df['spaces_left'].diff(12)
    df['delta_24h'] = df['spaces_left'].diff(288)

    epsilon = 1e-6
    shift_3h = df['spaces_left'].shift(36)
    df['trend_3h'] = (df['spaces_left'] - shift_3h) / (shift_3h + epsilon)

    shift_24h = df['spaces_left'].shift(288)
    df['trend_24h'] = (df['spaces_left'] - shift_24h) / (shift_24h + epsilon)
    return df

In [ ]:
df_parking_architektura_5min = add_other_features(df_parking_architektura_5min)
df_parking_wronskiego_5min = add_other_features(df_parking_wronskiego_5min)
df_parking_polinka_5min = add_other_features(df_parking_polinka_5min)
df_parking_D20_5min = add_other_features(df_parking_D20_5min)
df_parking_geocentrum_5min = add_other_features(df_parking_geocentrum_5min)

In [ ]:
df_parking_architektura_5min.head(5)

In [ ]:

fig, axes = plt.subplots(5, 1, figsize=(50, 50), sharex=True)

sns.lineplot(data=df_parking_architektura_5min, x="measured_at", y="roll_mean_24h", ax=axes[0])
axes[0].set_title("Parking architektura roll_mean_24h")

sns.lineplot(data=df_parking_wronskiego_5min, x="measured_at", y="roll_mean_24h", ax=axes[1])
axes[1].set_title("Parking wronskiego roll_mean_24h")

sns.lineplot(data=df_parking_polinka_5min, x="measured_at", y="roll_mean_24h", ax=axes[2])
axes[2].set_title("Parking polinka roll_mean_24h")

sns.lineplot(data=df_parking_D20_5min, x="measured_at", y="roll_mean_24h", ax=axes[3])
axes[3].set_title("Parking D20 roll_mean_24h")

sns.lineplot(data=df_parking_geocentrum_5min, x="measured_at", y="roll_mean_24h", ax=axes[4])
axes[4].set_title("Parking geocentruym roll_mean_24h")

plt.show()

Roll mean smooths out very sudden dips and helps with data instability. It helps to creally visualise sumer break and weekly cycles

In [ ]:
parking_lots = [
    ("Parking Architektura", df_parking_architektura_5min),
    ("Parking Wronskiego", df_parking_wronskiego_5min),
    ("Parking Polinka", df_parking_polinka_5min),
    ("Parking D20", df_parking_D20_5min),
    ("Parking Geocentrum", df_parking_geocentrum_5min)
]

fig = make_subplots(rows=5, cols=1, subplot_titles=[p[0] for p in parking_lots])

for i, (title, df) in enumerate(parking_lots):

    fig.add_trace(
        go.Scatter(x=df.index, y=df['spaces_left'], name=f"{title} - Actual", 
                   line=dict(width=1)),
        row=i+1, col=1
    )
    

    fig.add_trace(
        go.Scatter(x=df.index, y=df['lag_7_days'], name=f"{title} - Lag 7d", 
                   line=dict(width=1, dash='dot')), # Dotted line for comparison
        row=i+1, col=1
    )

fig.update_layout(height=2000, width=1600, title_text="Parking Availability Analysis")
fig.show()

Lag features have great correlation with target value. Data has strong weekly seasonality.

In [ ]:
def plot_parking_heatmaps(parking_data_list):

    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

    n_plots = len(parking_data_list)
    fig, axes = plt.subplots(n_plots, 1, figsize=(15, 4 * n_plots), sharex=True)
    
    if n_plots == 1:
        axes = [axes]

    for i, (title, df) in enumerate(parking_data_list):
        df_temp = df.copy()
        
        # Extract Hour and Day Name from the index
        df_temp['hour'] = df_temp.index.hour
        df_temp['day'] = df_temp.index.day_name()
        
        heatmap_data = df_temp.groupby(['day', 'hour'])['spaces_left'].mean().unstack()
        
        heatmap_data = heatmap_data.reindex(days_order)
        
        sns.heatmap(
            heatmap_data, 
            cmap="icefire", 
            linewidths=.5, 
            ax=axes[i],
            cbar_kws={'label': 'Avg Spaces Left'} 
        )
        
        axes[i].set_title(title)
        axes[i].set_ylabel("Day of Week")
        
        if i == n_plots - 1:
            axes[i].set_xlabel("Hour of Day")
        else:
            axes[i].set_xlabel("")

    plt.tight_layout()
    plt.show()

plot_parking_heatmaps(parking_lots)

From heatmaps it can be crealy seen that hours between 6:00 and 15:00 on workdays have the most cars parked, with exception of "D20" parking lot "wronskiego and"polinka" parking lots also have som traffic on Sunday and saturday but not as mutch as D20. "Geocentrum" and "Architektura" have almost no cars on the weekend.  

# SKS EDA

SKS operating hours 

Godziny:
poniedziałek	07:30–17:00
wtorek	07:30–17:00
środa	07:30–17:00
czwartek	07:30–17:00
piątek	07:30–16:00
sobota	Zamknięte
niedziela	Zamknięte

In [ ]:
df_sks_users.shape

In [ ]:
df_sks_users = df_sks_users.sort_values(by=['external_timestamp'])
df_sks_users = df_sks_users.reset_index(drop=True)
df_sks_users.head(10)

Here external timestap predates created_at witch is weird and can suggest some error when logging data. 


### Adding features

In [ ]:
df_sks_users['day'] = df_sks_users['external_timestamp'].dt.day_name()

df_sks_users[['external_timestamp', 'day']].tail()

In [ ]:
def is_open_sks(row):

    day = row['day']
    time = row['external_timestamp'].time()
    if day in ['Saturday', 'Sunday']:
        return False
    elif day == 'Friday':
        # check if between 7:30 and 16:00
        if dt.time(7, 30) <= time < dt.time(16, 00):
            return True
        else:
            return False
    else:
         # check if between 7:30 and 17:00
        if dt.time(7, 30) <= time < dt.time(17, 00):
            return True
        else:
            return False
    

In [ ]:
df_sks_users['is_open'] = df_sks_users.apply(is_open_sks, axis=1)

In [ ]:
event_intervals = pd.IntervalIndex.from_arrays(
    df_events['start'], 
    df_events['end'], 
)
df_sks_users['has_event'] = df_sks_users['external_timestamp'].apply(lambda x: event_intervals.contains(x).any())

### Check data anomlaites, duplicates, etc. 

In [ ]:
check_missing_logs(df_sks_users, 'external_timestamp', 300, 120)

In [ ]:
df_sks_users.iloc[58320:58330]

Very few gaps compared to parking_data but 24 day gap is extremly large and will be hard to fix. Perhaps this is a good plecae to try model based imputation ? 

In [ ]:
value_counts = df_sks_users["created_at"].value_counts()
duplicates = value_counts[value_counts > 1]
duplicates.head(100)

In [ ]:
value_counts = df_sks_users["created_at"].value_counts()
duplicates = value_counts[value_counts > 1]
duplicates.head(20)

A day consists of 1440 minutes (24 hours * 60 minutes). Divided by 5, there should be 288 records. It seems all timestamps for a day are created at this interval, but sometimes there are 292 records. This equals 1460 minutes (292 * 5), which is 20 minutes more than there are in a day. This can be attributed to an error. The usefulness of features created_at and updated_at leaves a lot to be desired; external_timestamp carries the more important information.

In [ ]:
value_counts = df_sks_users["external_timestamp"].value_counts()
duplicates = value_counts[value_counts > 1]
print(duplicates)

Thre are no duplicte values in external_timestamp. 

In [ ]:
plt.figure(figsize=(50, 10))

sns.lineplot(data=df_sks_users, x="external_timestamp", y="active_users", hue="has_event")

plt.show()

has_event does explain some anomalities in data but beacuse of number of "less important" events it has low usability, it highlights many days but only tiwce does it hit anomlity and it even misses one. 

In [ ]:
plt.figure(figsize=(50, 10))

sns.lineplot(data=df_sks_users, x="external_timestamp", y="active_users", hue="is_open")

plt.show()

is_open is strongy corealted with number of active users

In [ ]:
plt.figure(figsize=(50, 10))

sns.lineplot(data=df_sks_users, x="external_timestamp", y="moving_average_21")

plt.show()

I don't see exactly what 'moving_average_21' represents. The name suggests moving average from the last 21 minutes / hours / logs but i did not manage to recreate this feature. 

### Resampling to fill in gaps


In [ ]:
agg_rules = {
    'active_users': 'mean',
    'moving_average_21': 'mean',
    'day': 'first', 
    'is_open': 'max',
    'has_event': 'max'
}

df_sks_users_5min = df_sks_users.resample('5min', on='external_timestamp').agg(agg_rules)
df_sks_users_5min.head(20)

In [ ]:
df_sks_users_5min.shape

In [ ]:
df_sks_users.shape

In [ ]:
df_sks_users_5min['lag_1_hour'] = df_sks_users_5min['active_users'].shift(12) # 12 * 5 = 60 min 
df_sks_users_5min['lag_3_hours'] = df_sks_users_5min['active_users'].shift(36)
df_sks_users_5min['lag_1_day'] = df_sks_users_5min['active_users'].shift(288) # 24 * 60 / 5 = 288
df_sks_users_5min['lag_7_days'] = df_sks_users_5min['active_users'].shift(288*7)
df_sks_users_5min['lag_2_weeks'] = df_sks_users_5min['active_users'].shift(288*7*2)

df_sks_users_5min[['active_users','lag_1_hour', 'lag_3_hours', 'lag_1_day', 'lag_7_days','lag_2_weeks']].iloc[18500:18788]

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(50, 20), sharex=True)

sns.lineplot(data=df_sks_users_5min, x="external_timestamp", y="active_users",ax=axes[0])
sns.lineplot(data=df_sks_users_5min, x="external_timestamp", y="lag_7_days", ax=axes[0])
axes[0].set_title("Active users lag 7 days")

sns.lineplot(data=df_sks_users_5min, x="external_timestamp", y="active_users",ax=axes[1])
sns.lineplot(data=df_sks_users_5min, x="external_timestamp", y="lag_2_weeks", ax=axes[1])
axes[1].set_title("Active users lag 2 weeks")

plt.show()

Based on chart data, poses strong weekly seasonality, especially 7-day lag seems like a very useful feature. 

In [ ]:
df_sks_users_5min['roll_mean_3h'] = df_sks_users_5min['active_users'].rolling(window=36, closed='left').mean()
df_sks_users_5min['roll_mean_24h'] = df_sks_users_5min['active_users'].rolling(window=288, closed='left').mean()

df_sks_users_5min['delta_1h'] = df_sks_users_5min['active_users'].diff(12)
df_sks_users_5min['delta_24h'] = df_sks_users_5min['active_users'].diff(288)

epsilon = 1e-6
shift_3h = df_sks_users_5min['active_users'].shift(36)
df_sks_users_5min['trend_3h'] = (df_sks_users_5min['active_users'] - shift_3h) / (shift_3h + epsilon)

shift_24h = df_sks_users_5min['active_users'].shift(288)
df_sks_users_5min['trend_24h'] = (df_sks_users_5min['active_users'] - shift_24h) / (shift_24h + epsilon)

df_sks_users_5min[['active_users', 'roll_mean_3h','roll_mean_24h', 'delta_1h','delta_24h', 'trend_3h','trend_24h']].iloc[18500:18788]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(50, 20), sharex=True)

sns.lineplot(data=df_sks_users_5min, x="external_timestamp", y="roll_mean_3h", ax=axes[0])
axes[0].set_title("3h Rolling Mean")

sns.lineplot(data=df_sks_users_5min, x="external_timestamp", y="roll_mean_24h", ax=axes[1])
axes[1].set_title("24h Rolling Mean")

plt.show()

In [ ]:
df_sks_users_5min['hour'] = df_sks_users_5min.index.hour

heatmap_data = df_sks_users_5min.groupby(['day', 'hour'])['active_users'].mean().unstack()

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days_order)

plt.figure(figsize=(15, 6))
sns.heatmap(heatmap_data, cmap="icefire", linewidths=.5)
plt.title("Heatmap: Activity by Day and Hour")
plt.ylabel("Day of Week")
plt.xlabel("Hour of Day")
plt.show()

Heatmap suggests that 8:00 to 13:00 on working days sees the highest activity. There is a surprising amount of activity on Thursday and Friday after 19:00, suggests extra activities in SKS after lectures are over. Thursday could be correlated with board games club in that regard. 

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(data=df_sks_users_5min, x='hour', y='active_users', color='skyblue')

plt.title('Distribution by Hour of Day')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()